# Tutorial 6: Async Operations with Futures

Use Python's `async` / `await` to read numeric entries from S3, double
each value, and write the results back — all without blocking the event
loop. This tutorial introduces three async patterns:

1. **`await` on a `GroupFuture`** — the future returned by batch
   `memorize` / `remember` calls is directly awaitable.
2. **`await` on an individual future** via the policy's future bank.
3. **`async with laila.guarantee_async:`** — an async context manager
   that automatically tracks and awaits every future created inside it.

**Prerequisites:** `pip install "laila-core[s3]"` and a `secrets.toml` with your AWS credentials.

In [ ]:
import asyncio

import laila
from laila.pool import S3Pool

## Load credentials and register the S3 pool

Create a `secrets.toml` in the same directory:
```toml
AWS_BUCKET_NAME = "your-bucket"
AWS_ACCESS_KEY_ID = "AKIA..."
AWS_SECRET_ACCESS_KEY = "wJa..."
AWS_REGION = "us-east-1"
```

In [ ]:
laila.read_args("./secrets.toml")

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
    nickname="async_pool",
)
laila.memory.extend(s3_pool, pool_nickname="async_pool")

## Step 1: Create and upload numeric entries

Wrap the integers 1 through 10 as LAILA constant entries and push them
to S3. `memorize` with a list returns a `GroupFuture` — we `await` it
directly instead of calling `laila.wait`.

In [ ]:
entries = [laila.constant(data=i, nickname=f"number_{i}") for i in range(1, 11)]

upload_future = laila.memorize(entries, pool_nickname="async_pool")
print("Upload status before await:", laila.status(upload_future))

await upload_future

print("Upload status after await: ", laila.status(upload_future))
print(f"Uploaded {len(entries)} entries to S3")

## Step 2: Define the async doubling function

The core of the tutorial: an `async` function that remembers entries
from S3, doubles every value, and writes the results back. Each I/O
call returns a future that we `await`, so the event loop stays free
between operations.

In [ ]:
async def double_entries(entry_ids, pool_nickname):
    """Remember entries from S3, double each value, and store the results back."""
    remember_future = laila.remember(entry_ids, pool_nickname=pool_nickname)
    remembered = await remember_future

    doubled = []
    for entry in remembered:
        new_entry = laila.constant(
            data=entry.data * 2,
            nickname=f"doubled_{entry.nickname}",
        )
        doubled.append(new_entry)

    upload_future = laila.memorize(doubled, pool_nickname=pool_nickname)
    await upload_future

    return doubled

## Step 3: Run the async event loop

Jupyter runs its own `asyncio` event loop, so top-level `await` works
out of the box. Call `double_entries` and print the original vs doubled
values side by side.

In [ ]:
original_ids = [e.global_id for e in entries]

doubled_entries = await double_entries(original_ids, pool_nickname="async_pool")

print(f"{'Original':>10}  {'Doubled':>10}")
print("-" * 23)
for orig, dbl in zip(entries, doubled_entries):
    print(f"{orig.data:>10}  {dbl.data:>10}")

## Step 4: Verify the results from S3

Delete local references and recall the doubled entries purely by their
`global_id`. This proves the values actually round-tripped through S3.

In [ ]:
doubled_ids = [e.global_id for e in doubled_entries]
del doubled_entries

verify_future = laila.remember(doubled_ids, pool_nickname="async_pool")
verified = await verify_future

print("Verified doubled values from S3:")
for i, entry in enumerate(verified, start=1):
    expected = i * 2
    assert entry.data == expected, f"Expected {expected}, got {entry.data}"
    print(f"  number_{i}: {i} -> {entry.data}")

print("\nAll assertions passed.")

## Step 5: Reactive processing with `guarantee_async`

`laila.guarantee_async` is an async context manager that tracks every
future created inside its scope and awaits them all on exit. This is
useful for a reactive-style loop: process entries one at a time and let
the guarantee handle completion.

In [ ]:
quadrupled_entries = []
bank = laila.get_active_policy().future_bank

async with laila.guarantee_async:
    for gid in doubled_ids:
        ref = laila.remember(gid, pool_nickname="async_pool")
        recalled = await bank[ref.global_id]

        quad_entry = laila.constant(
            data=recalled.data * 2,
            nickname=f"quadrupled_{recalled.nickname}",
        )
        laila.memorize(quad_entry, pool_nickname="async_pool")
        quadrupled_entries.append(quad_entry)

        print(f"  {recalled.data:>4} -> {quad_entry.data:>4}")

print(f"\nProcessed {len(quadrupled_entries)} entries inside guarantee_async")

## Step 6: Inspect futures

Every future created by LAILA is stored in the active policy's **future
bank**. You can query status at any time with `laila.status(future)`,
which returns a percentage breakdown for `GroupFuture` objects.

In [ ]:
print("upload_future status:", laila.status(upload_future))
print(f"  children: {len(upload_future)}")
print()
print("verify_future status:", laila.status(verify_future))
print(f"  children: {len(verify_future)}")
print()

future_bank = laila.get_active_policy().future_bank
print(f"Total futures in bank: {len(future_bank)}")

## Clean up

In [ ]:
all_ids = (
    original_ids
    + doubled_ids
    + [e.global_id for e in quadrupled_entries]
)

async with laila.guarantee_async:
    laila.forget(all_ids, pool_nickname="async_pool")

print(f"Cleaned up {len(all_ids)} entries from S3")

## Summary

- `laila.memorize` and `laila.remember` with lists return a **`GroupFuture`** — it is directly awaitable via `await future`
- For a single-entry operation, look up the future in the **future bank** (`laila.get_active_policy().future_bank`) and `await` that
- **`async with laila.guarantee_async:`** tracks every future created in its scope and awaits them all on exit — ideal for reactive loops where entries arrive one at a time
- `GroupFuture.__await__` uses `asyncio.gather` internally, so all children resolve concurrently
- `laila.status(future)` returns a percentage breakdown (`finished`, `running`, `error`, etc.) at any point
- This pattern generalises to any async data pipeline — transform, filter, enrich, or route entries without blocking